# Aula 7 – Limpeza de Dados – Parte 1 (Nulos, Duplicados, Tipos)

## 1. Objetivos da aula
Ao final desta aula, você será capaz de:

* Detectar valores ausentes (nulos) em um DataFrame.
* Remover ou preencher valores ausentes usando `dropna` e `fillna`.
* Identificar e remover linhas duplicadas com `duplicated` e `drop_duplicates`.
* Analisar e corrigir tipos de dados com `dtypes`, `astype`, `to_datetime` e `to_numeric`.
* Montar um fluxo básico de limpeza inicial para um dataset real.

Esta é a aula em que os dados começam a ficar com “cara de dados de verdade”: menos sujeira, mais consistência.

## 2. Por que limpar dados?
Na prática, é muito raro receber um arquivo perfeito. Os problemas mais comuns:

* Campos em branco (nulos).
* Linhas repetidas (duplicadas).
* Colunas numéricas armazenadas como texto.
* Datas em formato errado.
* Valores “quebrados” por erros de digitação ou exportação.

Se você não limpar os dados as estatísticas ficam erradas, os gráficos ficam distorcidos e os modelos de machine learning aprendem coisas falsas. Pandas oferece ferramentas para tratar esses problemas de forma sistemática.

## 3. Dataset exemplo: Cadastro de clientes com buracos
Vamos criar um DataFrame que simula um cadastro de clientes, com alguns problemas:

In [2]:
import pandas as pd
import numpy as np

dados_clientes = {
    "id_cliente": [1, 2, 3, 4, 4],
    "nome": ["Ana", "Bruno", "Carla", "Daniel", "Daniel"],
    "idade": [25, np.nan, 32, 40, 40],
    "renda": [3000.0, 7000.0, np.nan, 12000.0, 12000.0],
    "cidade": ["Juiz de Fora", "São Paulo", None, "Rio de Janeiro", "Rio de Janeiro"]
}

df_clientes = pd.DataFrame(dados_clientes)
df_clientes

,id_cliente,nome,idade,renda,cidade
0,1,Ana,25.0,3000.0,Juiz de Fora
1,2,Bruno,NaN,7000.0,São Paulo
2,3,Carla,32.0,NaN,NaN
3,4,Daniel,40.0,12000.0,Rio de Janeiro
4,4,Daniel,40.0,12000.0,Rio de Janeiro


Repare:
* Há `np.nan` em idade e renda.
* Há `None` em cidade (também tratado como nulo).
* Há uma linha duplicada (`id_cliente=4`, Daniel, mesma renda, mesma cidade).

## 4. Identificando valores nulos
Valores nulos podem vir como `NaN` (Not a Number), `None` ou strings vazias. O pandas trata `NaN` e `None` como nulos em colunas numéricas/objetos.

### 4.1. Verificar nulos com isna ou isnull
`isna()` e `isnull()` são equivalentes.

In [3]:
df_clientes.isna()

,id_cliente,nome,idade,renda,cidade
0,False,False,False,False,False
1,False,False,True,False,False
2,False,False,False,True,True
3,False,False,False,False,False
4,False,False,False,False,False


Mais útil é somar por coluna:

In [4]:
df_clientes.isna().sum()

id_cliente    0
nome          0
idade         1
renda         1
cidade        1
dtype: int64

### 4.2. Verificar não nulos com notna

In [5]:
df_clientes.notna().sum()

id_cliente    5
nome          5
idade         4
renda         4
cidade        4
dtype: int64

## 5. Removendo valores nulos com `dropna` 
Em alguns cenários, você pode simplesmente remover linhas com nulos. Mas é preciso cuidado: retirar linhas significa perder informação.

### 5.1. Remover linhas com qualquer nulo

In [6]:
df_sem_nulos = df_clientes.dropna()
df_sem_nulos

,id_cliente,nome,idade,renda,cidade
0,1,Ana,25.0,3000.0,Juiz de Fora
3,4,Daniel,40.0,12000.0,Rio de Janeiro
4,4,Daniel,40.0,12000.0,Rio de Janeiro


### 5.2. Remover linhas com nulos em colunas específicas
Exemplo: você só quer remover linhas se renda for nulo, mas não se cidade estiver em branco.

In [7]:
df_sem_renda_nula = df_clientes.dropna(subset=["renda"])
df_sem_renda_nula

,id_cliente,nome,idade,renda,cidade
0,1,Ana,25.0,3000.0,Juiz de Fora
1,2,Bruno,NaN,7000.0,São Paulo
3,4,Daniel,40.0,12000.0,Rio de Janeiro
4,4,Daniel,40.0,12000.0,Rio de Janeiro


## 6. Preenchendo valores nulos com `fillna` 
Em vez de remover linhas, muitas vezes é melhor preencher valores nulos com algo razoável (média, mediana, valor fixo).

### 6.1. Preencher com valor fixo

In [8]:
df_clientes["cidade"].fillna("Desconhecida", inplace=True)
df_clientes

/tmp/ipykernel_9357/161176113.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_clientes["cidade"].fillna("Desconhecida", inplace=True)


,id_cliente,nome,idade,renda,cidade
0,1,Ana,25.0,3000.0,Juiz de Fora
1,2,Bruno,NaN,7000.0,São Paulo
2,3,Carla,32.0,NaN,NaN
3,4,Daniel,40.0,12000.0,Rio de Janeiro
4,4,Daniel,40.0,12000.0,Rio de Janeiro


### 6.2. Preencher com a média ou mediana da coluna

In [9]:
media_renda = df_clientes["renda"].mean()
df_clientes["renda"].fillna(media_renda, inplace=True)

mediana_idade = df_clientes["idade"].median()
df_clientes["idade"].fillna(mediana_idade, inplace=True)

df_clientes

/tmp/ipykernel_9357/1612340511.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_clientes["renda"].fillna(media_renda, inplace=True)
/tmp/ipykernel_9357/1612340511.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never wo

,id_cliente,nome,idade,renda,cidade
0,1,Ana,25.0,3000.0,Juiz de Fora
1,2,Bruno,NaN,7000.0,São Paulo
2,3,Carla,32.0,NaN,NaN
3,4,Daniel,40.0,12000.0,Rio de Janeiro
4,4,Daniel,40.0,12000.0,Rio de Janeiro


Depois disso, teste novamente:

In [10]:
df_clientes.isna().sum()

id_cliente    0
nome          0
idade         1
renda         1
cidade        1
dtype: int64

## 7. Identificando e removendo duplicados
Duplicados são linhas que aparecem exatamente iguais em todas as colunas, ou em um subconjunto relevante.

### 7.1. Checando duplicados com duplicated

In [11]:
df_clientes.duplicated()

0    False
1    False
2    False
3    False
4     True
dtype: bool

Para ver apenas as duplicadas:

In [12]:
df_clientes[df_clientes.duplicated()]

,id_cliente,nome,idade,renda,cidade
4,4,Daniel,40.0,12000.0,Rio de Janeiro


Você também pode verificar duplicados com base em uma coluna, como `id_cliente`:

In [13]:
df_clientes.duplicated(subset=["id_cliente"])

0    False
1    False
2    False
3    False
4     True
dtype: bool

### 7.2. Removendo duplicados com drop_duplicates

In [14]:
df_sem_duplicatas = df_clientes.drop_duplicates()
df_sem_duplicatas

,id_cliente,nome,idade,renda,cidade
0,1,Ana,25.0,3000.0,Juiz de Fora
1,2,Bruno,NaN,7000.0,São Paulo
2,3,Carla,32.0,NaN,NaN
3,4,Daniel,40.0,12000.0,Rio de Janeiro


Para remover duplicados com base apenas em algumas colunas (por exemplo, `id_cliente`):

In [15]:
df_sem_duplicatas_id = df_clientes.drop_duplicates(subset=["id_cliente"])
df_sem_duplicatas_id

,id_cliente,nome,idade,renda,cidade
0,1,Ana,25.0,3000.0,Juiz de Fora
1,2,Bruno,NaN,7000.0,São Paulo
2,3,Carla,32.0,NaN,NaN
3,4,Daniel,40.0,12000.0,Rio de Janeiro


## 8. Tipos de dados: verificando e entendendo `dtypes` 
Além de nulos e duplicados, outro problema comum é o tipo errado.

In [16]:
dados_pedidos = {
    "id_pedido": ["1", "2", "3", "4"],
    "data_pedido": ["2024-10-01", "2024-10-02", "2024/10/03", "2024-10-04"],
    "valor": ["200.50", "350,00", "150.00", "abc"]
}

df_pedidos = pd.DataFrame(dados_pedidos)
df_pedidos

,id_pedido,data_pedido,valor
0,1,2024-10-01,200.50
1,2,2024-10-02,"350,00"
2,3,2024/10/03,150.00
3,4,2024-10-04,abc


In [17]:
df_pedidos.dtypes

id_pedido      str
data_pedido    str
valor          str
dtype: object

## 9. Convertendo tipos com `astype`, `to_datetime` e `to_numeric`

### 9.1. Convertendo datas com to_datetime

In [18]:
df_pedidos["data_pedido"] = pd.to_datetime(df_pedidos["data_pedido"], errors="coerce")
df_pedidos

,id_pedido,data_pedido,valor
0,1,2024-10-01,200.50
1,2,2024-10-02,"350,00"
2,3,NaT,150.00
3,4,2024-10-04,abc


Parâmetros importantes:
* `errors="raise"` (padrão): lança erro.
* `errors="coerce"`: valores inválidos viram `NaT` (nulo para datas).
* `errors="ignore"`: deixa como está.

### 9.2. Convertendo números com to_numeric

In [19]:
# Primeiro, corrigir vírgula para ponto
df_pedidos["valor_limpo"] = df_pedidos["valor"].str.replace(",", ".", regex=False)

# Converter para numérico
df_pedidos["valor_num"] = pd.to_numeric(df_pedidos["valor_limpo"], errors="coerce")
df_pedidos

,id_pedido,data_pedido,valor,valor_limpo,valor_num
0,1,2024-10-01,200.50,200.50,200.5
1,2,2024-10-02,"350,00",350.00,350.0
2,3,NaT,150.00,150.00,150.0
3,4,2024-10-04,abc,abc,NaN


`errors="coerce"` transforma valores não numéricos em `NaN`. Depois disso, você pode decidir remover linhas ou preencher com 0.

### 9.3. Convertendo tipos simples com astype

In [20]:
df_pedidos["id_pedido"] = df_pedidos["id_pedido"].astype("int64")

## 10. Montando um mini fluxo de limpeza
Dado um DataFrame qualquer, um fluxo de limpeza inicial poderia ser:

1. Verificar formas e tipos (`shape`, `dtypes`).
2. Verificar nulos (`isna().sum()`).
3. Tomar decisões (quais colunas podem ter nulos, quais são obrigatórias).
4. Remover duplicados (`drop_duplicates`).
5. Converter tipos (`to_datetime`, `to_numeric`).
6. Preencher nulos (mediana/média ou valores padrão).

## 11. Resumo da aula
Nesta aula, você aprendeu a:

* **Encontrar valores nulos** com `isna` e `sum`.
* **Remover nulos** com `dropna`.
* **Preencher nulos** com `fillna`.
* **Identificar e remover duplicados** com `duplicated` e `drop_duplicates`.
* **Verificar e converter tipos de dados** com `dtypes`, `to_datetime` e `to_numeric`.

Essas técnicas formam a base da limpeza de dados. Na próxima parte, vamos aprofundar com outliers, categorias e datas em séries temporais.